# Notebook 2 - High-Water Mark to the Rescue

In notebook 1 we saw a write get acknowledged and then lost. Now we fix it.

## Definition

> The **high-water mark (HWM)** is the highest log offset that has been replicated to a **quorum** of nodes (leader + enough followers). Clients can only read entries with offset <= HWM. Anything above is *tentative*.

A *quorum* is "more than half" - for 3 nodes that is 2, for 5 nodes that is 3. As long as a quorum survives, the next leader will have every committed entry, so committed reads are safe across failovers.

In this notebook we:

1. Build a small leader that tracks **per-follower replicated offsets**.
2. Compute the HWM after every replication event.
3. Crash the leader and verify that everything a client saw still exists.
4. Plot how the HWM advances over time.


## Setup (same as notebook 1)

```bash
cd 02-distributed-primitives/high-water-mark
uv sync
```

Pick the `.venv` kernel in VS Code, reload the window if it is missing.


## BETTER: track which followers have which offset

The leader maintains a table `match_index[follower] = highest offset that follower has`. When followers send acknowledgments, the table gets updated. The HWM is recomputed as *"the highest offset that >= quorum nodes (counting the leader) have"*.


In [ ]:
from dataclasses import dataclass, field
from typing import List, Dict

@dataclass
class Node:
    name: str
    log: List[str] = field(default_factory=list)

@dataclass
class Leader:
    name: str = "L"
    log: List[str] = field(default_factory=list)
    # follower name -> highest replicated offset (-1 means "nothing yet")
    match_index: Dict[str, int] = field(default_factory=dict)
    quorum: int = 2  # for 3 total nodes, 2 = majority

    def append(self, entry: str) -> int:
        self.log.append(entry)
        return len(self.log) - 1  # offset

    def ack(self, follower: str, offset: int) -> None:
        self.match_index[follower] = max(self.match_index.get(follower, -1), offset)

    @property
    def high_water_mark(self) -> int:
        # The leader implicitly has every entry it appended.
        leader_offset = len(self.log) - 1
        offsets = sorted([leader_offset, *self.match_index.values()], reverse=True)
        # offsets[quorum-1] = the offset that 'quorum' nodes are at-or-above.
        return offsets[self.quorum - 1] if offsets else -1

    def committed_view(self) -> List[str]:
        return self.log[: self.high_water_mark + 1]


## Replicate three writes with different delivery patterns

We will record `(entry, hwm, committed_view)` after each step so we can plot it.


In [ ]:
leader = Leader()
f1, f2 = Node("f1"), Node("f2")

history = []  # list of (label, hwm, committed_view)

def replicate(entry, delivered_to):
    off = leader.append(entry)
    for f in delivered_to:
        f.log.append(entry)
        leader.ack(f.name, off)
    history.append((entry, leader.high_water_mark, list(leader.committed_view())))
    print(
        f"append {entry!r} off={off}  hwm={leader.high_water_mark}  "
        f"committed={leader.committed_view()}"
    )

replicate("A", [f1, f2])  # both followers ack -> committed
replicate("B", [f1])      # f1 acks; leader+f1 = quorum -> committed
replicate("C", [])        # nobody acks -> NOT committed (HWM stays put)


`C` is in the leader's local log but **not** in `committed_view()`. Clients reading via the leader will only see `[A, B]`, never `C`.

## Crash the leader and elect a new one


In [ ]:
client_view_before = leader.committed_view()
print("client saw before crash:", client_view_before)

# Promote the follower with the longest log.
# (In real Raft we would also tie-break on term, but the longest-log rule is enough here.)
new_leader = max([f1, f2], key=lambda f: len(f.log))
print("new leader   :", new_leader.name)
print("new log      :", new_leader.log)
print()
print(
    "all committed entries survived?",
    all(e in new_leader.log for e in client_view_before),
)


Every entry the client *saw* is still there. The uncommitted `C` is silently dropped - the right outcome, because the client was never told it succeeded.

## Visualizing the high-water mark

The HWM only advances when a quorum has caught up. Watch how it lags behind the leader's local log size.


In [ ]:
import matplotlib.pyplot as plt

labels   = [h[0] for h in history]
hwms     = [h[1] for h in history]
log_lens = list(range(len(history)))  # leader had 0, 1, 2 as its tail offset

fig, ax = plt.subplots(figsize=(6, 3.2))
ax.plot(range(len(history)), log_lens, marker="o", label="leader local tail offset")
ax.plot(range(len(history)), hwms,     marker="s", label="high-water mark")
ax.set_xticks(range(len(history)))
ax.set_xticklabels([f"append {l}" for l in labels])
ax.set_ylabel("offset")
ax.set_title("Leader tail vs. high-water mark")
ax.legend()
ax.grid(True, linestyle=":")
plt.tight_layout()
plt.show()


The gap between the two lines = entries that exist on the leader but **must not** be exposed to clients. That gap is what protects you from lost-write disasters.

## Takeaways

- HWM = highest offset replicated to **a quorum** (leader counts as one vote).
- Clients can only read entries with `offset <= HWM`.
- Followers learn the HWM from the leader (we will see how in notebook 3) and may **truncate** anything above it after a leader change.
- This single rule is the heart of Kafka's ISR model, Raft's `commitIndex`, and Multi-Paxos.
